# Project Alpha: RL Bot v63p

## 📋 TODO & Roadmap
- **Data Policy:** Should we apply `handle_zeros_as_nan` globally to all matrices?
- **Feature Scaling:** Finalize the `get_agent_view` dual-view scaling logic.
- **Performance:** Move heavy logic (`analyze_4d_regime`) into `core` modular library.

<details>
<summary><b>📜 Version History (v59 - v63)</b></summary>

### v63
- Added **Metric Registry** (Strong-typed, dual-view).
- Fixed **Index Poisoning** bug (Dates in Ticker column).
- Unified **Perception** and **Execution** in Audit packs.
- Added 4 New Pillars: Return Autocorr, Range Position, OBV Divergence, Convexity.

### v62
- Fixed **Object Reference Mutation** bug in `compute_alpha_ensemble`.
- Added `PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`.

### v61
- Verified all metrics with Excel parity.
- Refactored `AlphaEngine` into Orchestrator pattern.

### v60
- Converted notebook code to modular system.
- Added Volatility Regime plots.

### v59
- Implemented **Result Pattern** for error handling.
- Switched to logarithmic returns globally.
</details>

## I. Initialization & Environment

In [1]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import os
import gc
import pandas as pd
import numpy as np
import multiprocessing
import random
import re
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional
from types import SimpleNamespace
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput, MarketObservation
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.quant import QuantUtils, TickerEngine
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_DIR = SU.load_env_from_root("DATA_DIR")
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"✓ Paths Initialized.\nDATA Dir: {DATA_DIR}\nOHLCV: {DATA_PATH_OHLCV}\nIndices: {DATA_PATH_INDICES}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

✓ Paths Initialized.
DATA Dir: c:\Users\ping\Files_win10\python\py311\stocks\data\
OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
Indices: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


## II. Data Pipeline

In [2]:
# === RELOADING FROM PARQUET ===
features_df = pd.read_parquet("output/features_df.parquet")
macro_df = pd.read_parquet("output/macro_df.parquet")
df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

In [3]:
print("⏳ Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

print("⚡ Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

⏳ Loading DataFrames... takes ~6 minutes
⚡ Generating Features...
⚡ Generating Decoupled Features (Benchmark: SPY)...
🚀 Unstacking Wide Matrices...
✅ Pipeline Complete. Tickers: 1578, Days: 16194


In [4]:
# === SAVE TO PERSISTENCE SANDBOX ===
features_df.to_parquet("output/features_df.parquet", index=True)
macro_df.to_parquet("output/macro_df.parquet", index=True)
df_close_wide.to_parquet("output/df_close_wide.parquet", index=True)
df_atrp_wide.to_parquet("output/df_atrp_wide.parquet", index=True)
df_trp_wide.to_parquet("output/df_trp_wide.parquet", index=True)

## III. The Analysis Suite

In [5]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

🎯 Master AlphaEngine Ready.


In [6]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=3,
    debug=False,
)

analyzer1, _ = create_walk_forward_analyzer(engine, _inputs, universe_subset=None)
analyzer1.show()

DEBUG: 941 stocks passed filters on 2026-04-16


In [ ]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [8]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=3,
    debug=True,
)

print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df)
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
DEBUG: 941 stocks passed filters on 2026-04-16
----------------------------------------------------------------------
Timeline: [2026-04-01] -> Decision: 2026-04-16 -> Entry: 2026-04-17 -> End: 2026-04-24
Selected Tickers (3):
SOXL, CRDO, NBIS
----------------------------------------------------------------------


,Full,Lookback,Holding
Metric,,,
Group Gain,0.6818,0.5047,0.1557
Benchmark Gain,0.0858,0.0684,0.0053
== Gain Delta,0.5960,0.4363,0.1504
Group Sharpe,23.1077,24.1712,38.7616
Benchmark Sharpe,10.9495,14.0962,2.3401
== Sharpe Delta,12.1582,10.0750,36.4215
Group Sharpe (ATRP),0.6272,0.6934,0.5230
Benchmark Sharpe (ATRP),0.3892,0.4683,0.0879
== Sharpe (ATRP) Delta,0.2380,0.2251,0.4351



Target Reward Signal: 0.155737


In [9]:
##################################
##################################
##################################

In [23]:
result = SU.visualize_analyzer_structure(analyzer=analyzer2)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(49,))
[  2]   📈 benchmark_series (shape=(49,))
[  3]   🧮 normalized_plot_data (shape=(49, 50))
[  4]   📂 tickers (len=50)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]     📄 index_3 (str)
[  9]     📄 index_4 (str)
[ 10]     📄 index_5 (str)
[ 11]     📄 index_6 (str)
[ 12]     📄 index_7 (str)
[ 13]     📄 index_8 (str)
[ 14]     📄 index_9 (str)
[ 15]     📄 index_10 (str)
[ 16]     📄 index_11 (str)
[ 17]     📄 index_12 (str)
[ 18]     📄 index_13 (str)
[ 19]     📄 index_14 (str)
[ 20]     📄 index_15 (str)
[ 21]     📄 index_16 (str)
[ 22]     📄 index_17 (str)
[ 23]     📄 index_18 (str)
[ 24]     📄 index_19 (str)
[ 25]     📄 index_20 (str)
[ 26]     📄 index_21 (str)
[ 27]     📄 index_22 (str)
[ 28]     📄 index_23 (str)
[ 29]     📄 index_24 (str)
[ 30]     📄 index_25 (str)
[ 31]     📄 index_26 (str)
[ 32]     📄 index_27 (str)
[ 33]     📄 index_28 (str)
[ 34] 

In [24]:
target_tickers = SU.peek(4, result)

 📍 INDEX: [4]
 🏷️  NAME:  tickers
 📂 PATH:  audit_pack -> tickers



['INTC',
 'USO',
 'SOXL',
 'TSEM',
 'SNDK',
 'NOK',
 'STM',
 'RVMD',
 'STX',
 'CIEN',
 'WDC',
 'BE',
 'APA',
 'ONTO',
 'PBR',
 'TTMI',
 'MPWR',
 'CVE',
 'ASX',
 'SOXX',
 'BP',
 'MU',
 'FIX',
 'SMH',
 'KLAC',
 'SU',
 'STLD',
 'LSCC',
 'LITE',
 'HAL',
 'EWT',
 'TER',
 'SATS',
 'GOOGL',
 'GOOG',
 'UTHR',
 'FTI',
 'ROST',
 'JBHT',
 'MKSI',
 'CMI',
 'CAT',
 'KEYS',
 'ADI',
 'JAZZ',
 'TD',
 'BG',
 'RY',
 'LRCX',
 'XBI']

In [12]:
SU.export_audit_to_excel(
    audit_pack=analyzer1.last_run, filename="Audit_Verification_Report.xlsx"
)

📂 [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx
✨ Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR/output/Audit_Verification_Report.xlsx')

In [13]:
##################################
##################################
##################################

In [14]:
import pathlib

# Set display width to prevent line wrapping
pd.set_option("display.width", 2000)

# Show all columns instead of truncated view
pd.set_option("display.max_columns", None)


def load_latest_finviz_parquet(data_dir: str) -> pd.DataFrame:
    path = pathlib.Path(data_dir)
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"

    files = list(path.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {path.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Usage
df_finviz = load_latest_finviz_parquet(DATA_DIR)
print(f"df_finviz:\n{df_finviz}")

DEBUG: Loading file: 2026-05-06_df_finviz_merged_stocks_etfs.parquet
df_finviz:
       No.                                            Company               Index                  Sector                        Industry Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M    P/E  Fwd P/E   PEG    P/S    P/B    P/C  P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %    EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %   ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Pri

In [26]:
tickers_in_df_finviz = [x for x in target_tickers if x not in ["USO", "SATS"]]
df_finviz.loc[tickers_in_df_finviz]

# df_finviz.loc[target_tickers]

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
INTC,21,Intel Corp,"NDX, S&P 500",Technology,Semiconductors,USA,NASD,"Technology, Semiconductors",567990.0,25,567990.0,NaN,NaN,1.08,10.56,5.10,17.32,NaN,22.18,6.52,0.04,NaN,8/7/2024,NaN,-0.63,0.21,NaN,NaN,NaN,NaN,NaN,7.18,-288.04,86.07,1.36,53760.00,-3170.00,1426.32,9.33,5020.00,4250.00,84.56,15.49,-0.01,59.44,5.96,3.40,1.36,144.27,-1.60,-3.01,-2.06,2.31,1.85,0.39,0.40,35.90,2.47,-5.90,13.4411,19.27,113.59,132.53,186.10,466.75,206.26,2.19,5.67,5.0173,6.87,6.04,43.33,90.67,172.99,2.29,178.14,2.29,495.89,18.97 - 110.48,2.29,33027.44,85.92,Apr 23/a,5/3/1973,Yes,Yes,85100.0,1.88,2.57,2.62,106.3600,1.46,155236591,85.33,108.15,110.93,113.50,106.58,113.01,4.49,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,23.1624,9473.0093,844.9227,10.9491,38.8216,5.8911,11.6895,76.4468,12.9584,8.0488,39.1345,6.2259,8.1008,26.6053,4.6259,4.8472,11.8925,2.4540,3.2269,6.1180,1.8016,2.7307,5.2023,1.6632
SOXL,137,Direxion Daily Semiconductor Bull 3X ETF,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, Equity - Leve...",18530.0,782,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.05,0.08,9/23/2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.14,11.88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.1856,40.59,193.28,211.81,241.40,1212.10,294.60,5.55,10.29,6.2044,9.13,7.48,52.87,117.54,229.29,12.62,319.66,12.62,1268.40,12.12 - 147.26,12.62,53202.71,82.14,-,3/11/2010,Yes,Yes,NaN,5.01,9.56,NaN,84.1400,0.75,63254599,NaN,144.16,157.94,166.00,149.06,165.85,15.05,Equity - Leveraged / Inverse,Equities (Stocks),0.75,43.0,18530.0,NaN,-29.88,NaN,-29.42,-12950.00,-41.14,988.47,117.84,30.39,"U.S., equity, semiconductors, leverage",155.5311,9473.0093,844.9227,13.7274,102.8002,13.9516,8.2890,18.6239,3.4582,10.5148,23.7469,4.9092,7.8425,14.9837,3.3927,4.0457,6.5861,1.9127,3.0618,4.7721,1.6435,3.0292,4.8462,1.6713
TSEM,534,Tower Semiconductor Ltd,-,Technology,Semiconductors,Israel,NASD,"Technology, Semiconductors",24240.0,641,24240.0,111.85,NaN,1.03,15.48,8.36,21.04,NaN,25.94,10.31,NaN,NaN,-,0.00,1.94,0.56,NaN,NaN,NaN,NaN,NaN,13.69,43.76,5.23,9.05,1570.00,220.47,14.37,0.10,112.53,104.85,93.18,6.18,0.00,73.20,1.69,4.21,1.53,4.41,6.89,7.91,7.22,6.48,5.51,0.05,0.06,23.23,12.40,14.08,-0.5229,8.88,13.63,78.82,156.92,504.77,84.70,0.85,14.65,6.7552,7.08,6.72,2.55,23.81,93.47,-5.29,100.01,-5.29,512.19,35.42 - 228.99,-66.02,16323.84,58.75,May 13/b,10/26/1994,Yes,Yes,NaN,-6.79,3.17,1.83,2.8800,0.79,2284142,177.60,225.53,232.67,232.67,214.01,216.87,-3.84,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [16]:
print(df_finviz.loc[target_tickers])

      No.                                   Company Index      Sector                   Industry         Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M     P/E  Fwd P/E   PEG    P/S    P/B    P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %   EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %  ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %  52W Low %   

In [17]:
def cleanup_ui():
    clear_output(wait=True)
    pio.renderers.default = None
    gc.collect()
    print("🧹 Memory Cleared.")


def active_engine_audit():
    gc.collect()
    engines = [obj for obj in gc.get_objects() if type(obj).__name__ == "AlphaEngine"]
    print(f"Active AlphaEngine Instances: {len(engines)}")


active_engine_audit()

Active AlphaEngine Instances: 1


In [18]:
# print(features_df.info())
# display(features_df.head())
# display(df_indices.tail())

In [19]:
################################
################################
################################
################################

In [20]:
SU.peek(0, result)

 📍 INDEX: [0]
 🏷️  NAME:  audit_pack
 📂 PATH:  audit_pack



EngineOutput(portfolio_series=Date
2026-04-01    1.0000
2026-04-02    1.0448
2026-04-06    1.0736
2026-04-07    1.1157
2026-04-08    1.2222
2026-04-09    1.2799
2026-04-10    1.3768
2026-04-13    1.4861
2026-04-14    1.6280
2026-04-15    1.6786
2026-04-16    1.6565
2026-04-17    1.6761
2026-04-20    1.7388
2026-04-21    1.7746
2026-04-22    1.8428
2026-04-23    1.8776
2026-04-24    1.9774
dtype: float64, benchmark_series=Date
2026-04-01    1.0000
2026-04-02    1.0009
2026-04-06    1.0056
2026-04-07    1.0061
2026-04-08    1.0317
2026-04-09    1.0377
2026-04-10    1.0370
2026-04-13    1.0471
2026-04-14    1.0599
2026-04-15    1.0682
2026-04-16    1.0708
2026-04-17    1.0838
2026-04-20    1.0816
2026-04-21    1.0745
2026-04-22    1.0854
2026-04-23    1.0812
2026-04-24    1.0896
dtype: float64, normalized_plot_data=Ticker        SOXL    CRDO    NBIS
Date                              
2026-04-01  1.0000  1.0000  1.0000
2026-04-02  1.0094  1.0577  1.0674
2026-04-06  1.0488  1.0682  1.1039
2

EngineOutput(portfolio_series=Date
2026-04-01    1.0000
2026-04-02    1.0448
2026-04-06    1.0736
2026-04-07    1.1157
2026-04-08    1.2222
2026-04-09    1.2799
2026-04-10    1.3768
2026-04-13    1.4861
2026-04-14    1.6280
2026-04-15    1.6786
2026-04-16    1.6565
2026-04-17    1.6761
2026-04-20    1.7388
2026-04-21    1.7746
2026-04-22    1.8428
2026-04-23    1.8776
2026-04-24    1.9774
dtype: float64, benchmark_series=Date
2026-04-01    1.0000
2026-04-02    1.0009
2026-04-06    1.0056
2026-04-07    1.0061
2026-04-08    1.0317
2026-04-09    1.0377
2026-04-10    1.0370
2026-04-13    1.0471
2026-04-14    1.0599
2026-04-15    1.0682
2026-04-16    1.0708
2026-04-17    1.0838
2026-04-20    1.0816
2026-04-21    1.0745
2026-04-22    1.0854
2026-04-23    1.0812
2026-04-24    1.0896
dtype: float64, normalized_plot_data=Ticker        SOXL    CRDO    NBIS
Date                              
2026-04-01  1.0000  1.0000  1.0000
2026-04-02  1.0094  1.0577  1.0674
2026-04-06  1.0488  1.0682  1.1039
2

In [21]:
features_df.loc["TSLA"]

,ATR,ATRP,TRP,RSI,Mom_21,Consistency,IR_63,Beta_63,DD_21,AutoCorr_15,Ret_1d,Range_Pos_20,Slope_P_5,Slope_V_5,Convexity,RollingStalePct,RollMedDollarVol,RollingSameVolCount
Date,,,,,,,,,,,,,,,,,,
2010-06-29,NaN,0.0000,0.0000,50.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-06-30,0.4747,0.2988,0.2988,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,-0.0025,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-01,0.4677,0.3194,0.2573,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,-0.0785,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-02,0.4552,0.3556,0.2286,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,-0.1257,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-06,0.4425,0.4120,0.2588,0.0000,NaN,0.0,NaN,1.0000,0.0000,0.0000,-0.1609,NaN,-0.1004,-0.6060,0.0000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-30,14.2693,0.0374,0.0434,51.8141,0.0266,0.6,-0.0898,1.6384,-0.0474,0.0782,0.0237,0.6162,0.0013,-0.0814,0.0060,0.0,2.8642e+10,0.0
2026-05-01,14.6087,0.0374,0.0487,56.1443,0.0251,0.6,-0.1041,1.6728,-0.0245,0.1042,0.0241,0.7438,0.0078,0.0769,0.0084,0.0,2.8534e+10,0.0
2026-05-04,14.2680,0.0364,0.0251,56.9111,0.0885,0.6,-0.0774,1.6831,-0.0202,0.0968,0.0043,0.7672,0.0133,0.5663,0.0121,0.0,2.8534e+10,0.0
